# RL

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
import json

from thesis_project.rl import report

# Main

## Baselines

In [ ]:
from thesis_project import rl
from thesis_project.dataset.features_market import FeatureSpec
from thesis_project.dataset.features_transforms import Identity

market: list[FeatureSpec] = [
    FeatureSpec(base_id="basis", transforms=[Identity()]),
]

dataset_config = rl.dataset.DatasetConfig(
    ticker="fbtp", n_jobs=1, contract_mode="round", market=market
)

dummy_dataset = rl.dataset.build_rl_dataset(dataset_config)

In [ ]:
env_config_serial = rl.env.EnvConfig(
    reset_mode="serial",
    price_mode="quoted",
    persistence_min=10,
    trajectory_min=0,
)

In [ ]:
env = rl.train.make_env(dataset=dummy_dataset, env_config=env_config_serial)

fixed_long_records = rl.evaluation.evaluate_model(
    predictor=rl.policies.FixedLongPolicy(),
    env=env,
)

In [ ]:
report.report({0: fixed_long_records}, metadata={"which": "both"})

In [ ]:
env = rl.train.make_env(dataset=dummy_dataset, env_config=env_config_serial)

fixed_short_records = rl.evaluation.evaluate_model(
    predictor=rl.policies.FixedShortPolicy(),
    env=env,
)

In [ ]:
report.report({0: fixed_short_records}, metadata={"which": "both"})

In [ ]:
import numpy as np
import pandas as pd

seed = 42
rng = np.random.default_rng(seed)
seeds = rng.integers(low=0, high=9999, size=10)

random_records: dict[int, pd.DataFrame] = {}
for seed in seeds:
    env = rl.train.make_env(dataset=dummy_dataset, env_config=env_config_serial)
    policy = rl.policies.RandomPolicy(seed=seed)
    random_records[seed] = rl.evaluation.evaluate_model(predictor=policy, env=env)

In [ ]:
report.report(random_records, metadata={"which": "both"})

## Report

In [ ]:
# NOTE:
# - Run 3 -> Mid Prices
# - Run 4 -> Quoted Prices

root = config.RES_EXP_DIR / TICKER / "rl"
run = 3
folder = f"run-{run:03d}"
path = root / folder

with open(path / "config.json", "r") as f:
    settings = json.load(f)

seeds = settings["seeds"]

In [ ]:
records = report.load_records(
    path,
    batch_size=512,
    clip_range=0.2,
    seeds=seeds,
    split="train",
)

In [ ]:
from thesis_project.dataset.data import BASE_FEATURES
from thesis_project.utils.io import load_filtered_parquet

pnl_mean = 76_439.38
pnl_std = 17_757.55
n_trades_mean = 786.20
n_trades_std = 366.64

max_spread = pnl_mean / n_trades_mean

spread_basis = records[seeds[0]].set_index("timestamp")["basis_spread"]

spread_ctd = load_filtered_parquet(BASE_FEATURES["ctd_spread"].path).loc[spread_basis.index]
spread_ctd *= 2_000_000 / 100
spread_fut = load_filtered_parquet(BASE_FEATURES["fut_spread"].path).loc[spread_basis.index]
spread_fut *= 100_000 / 100

In [ ]:
n = 22
print(f"{'Max spread per trade:':<{n}} {max_spread:,.2f} €")
print(f"{'Basis spread:':<{n}} {spread_basis.mean() / 2:,.2f} ± {spread_basis.std() / 2:,.2f} €")
print(f"{'FBTP CTD spread:':<{n}} {spread_ctd.mean() / 2:,.2f} ± {spread_ctd.std() / 2:,.2f} €")
print(
    f"{'FBTP FUT spread:':<{n}} {15.5 * spread_fut.mean() / 2:,.2f} ± {18 * spread_fut.std() / 2:,.2f} €"
)

In [ ]:
report.report(records)

In [ ]:
records = report.load_records(
    path,
    batch_size=512,
    clip_range=0.2,
    seeds=seeds,
    split="test",
)

In [ ]:
report.report(records)